In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy.ndimage import gaussian_filter
import pandas as pd
import scipy.optimize
import math
import MDAnalysis as md

def weighted_avg_and_std(values, weights):
    """
    Return the weighted average and standard deviation.

    They weights are in effect first normalized so that they 
    sum to 1 (and so they must not all be 0).

    values, weights -- NumPy ndarrays with the same shape.
    """
    average = np.average(values, weights=weights)
    # Fast and numerically precise:
    variance = np.average((values-average)**2, weights=weights)
    return (average, math.sqrt(variance))


path = '/Volumes/Elements/PTM_project/PARK7/TYR_COORD_METAD_chainA_BF40/'

In [2]:
u = md.Universe(path+'processed.pdb',path+'fit3.xtc') 

/Users/olivierstreit/miniconda3/lib/python3.7/site-packages/MDAnalysis/topology/guessers.py:80: UserWarning: Failed to guess the mass for the following atom types: 
  warnings.warn("Failed to guess the mass for the following atom types: {}".format(atom_type))
/Users/olivierstreit/miniconda3/lib/python3.7/site-packages/MDAnalysis/topology/PDBParser.py:330: UserWarning: Element information is absent or missing for a few atoms. Elements attributes will not be populated.
  warnings.warn("Element information is absent or missing for a few "


In [3]:
# pick 5 lowest free energy structures in the open and close state (SASA below or above 0.35 nm^2)

In [10]:

sim=3
T=300
t= 1000
t_discard = 200
frame = int(t*1000/5)
init_frame = int(t_discard*1000/5)


SASA = np.loadtxt(path+'RMSD_SASA_data/sasaTYR67_{}.xvg'.format(sim),skiprows=25)[:,2]
zeta = np.loadtxt(path+'DATA_PLUMED/COLVAR.{}'.format(sim))[:,1]
time = np.loadtxt(path+'DATA_PLUMED/COLVAR.{}'.format(sim))[:,0]
rbias = np.loadtxt(path+'DATA_PLUMED/COLVAR.{}'.format(sim))[:,3]


# FEL rbias reweighting
kbt = 0.008314*T
weights = np.exp((rbias[init_frame:frame]-np.amax(rbias[init_frame:frame]))/kbt)
weights = weights/np.sum(weights)
SASA=SASA[init_frame:frame]
zeta=zeta[init_frame:frame]
time=time[init_frame:frame]


frame = int(t*1000/5)
init_frame = int(t_discard*1000/5)
openframes = np.where(SASA>1.5)[0] 
closedframes = np.where(SASA<0.35)[0] 

# open state lowest free energy
N=5
ind_open = np.argpartition(weights[openframes], -N)[-N:]
# closed state
ind_closed = np.argpartition(weights[closedframes], -N)[-N:]

print(zeta[openframes][ind_open])
print(zeta[closedframes][ind_closed])

print(SASA[openframes][ind_open])
print(SASA[closedframes][ind_closed])


[-2.883417  0.938605 -1.144991 -1.968614  2.679257]
[-2.853723 -0.496268 -0.040513 -2.658075  2.830683]
[1.528 1.594 1.547 1.541 1.563]
[0. 0. 0. 0. 0.]


In [11]:
print(time[openframes][ind_open])
print(time[closedframes][ind_closed])

[867180.041189 335525.015937 207695.009865 338805.016092 866705.041166]
[457725.021741 456865.0217   456030.02166  457500.02173  457885.021748]


In [12]:


# open states
for TIME in time[openframes][ind_open]:
    FRAME =  int(TIME/5) # 5 ps per frame
    u.trajectory[FRAME]
    protein = u.select_atoms('all')
    with md.Writer('PARK7_chainA_Tyr67_sim{}_open_{}ps.pdb'.format(sim,int(TIME)),protein.n_atoms) as W:
        W.write(protein)

        
print('Done')

Done


In [13]:


# closed states
for TIME in time[closedframes][ind_closed]:
    FRAME =  int(TIME/5) # 5 ps per frame
    u.trajectory[FRAME]
    protein = u.select_atoms('all')
    with md.Writer('PARK7_chainA_Tyr67_sim{}_closed_{}ps.pdb'.format(sim,int(TIME)),protein.n_atoms) as W:
        W.write(protein)

        
print('Done')

Done


In [30]:
# find lowest free energy conformations of proline cis and trans states
sim=3
T=300
t= 1000
t_discard = 200
frame = int(t*1000/5)
init_frame = int(t_discard*1000/5)


SASA = np.loadtxt(path+'RMSD_SASA_data/sasaTYR67_{}.xvg'.format(sim),skiprows=25)[:,2]
zeta = np.loadtxt(path+'DATA_PLUMED/COLVAR.{}'.format(sim))[:,1]
time = np.loadtxt(path+'DATA_PLUMED/COLVAR.{}'.format(sim))[:,0]
rbias = np.loadtxt(path+'DATA_PLUMED/COLVAR.{}'.format(sim))[:,3]


# FEL rbias reweighting
kbt = 0.008314*T
weights = np.exp((rbias[init_frame:frame]-np.amax(rbias[init_frame:frame]))/kbt)
weights = weights/np.sum(weights)
SASA=SASA[init_frame:frame]
zeta=zeta[init_frame:frame]
time=time[init_frame:frame]


frame = int(t*1000/5)
init_frame = int(t_discard*1000/5)
cisframes = np.where(np.abs(zeta)<1.4)[0] 
transframes = np.where(np.abs(zeta)>1.4)[0] 

# open state lowest free energy
N=5
ind_cis = np.argpartition(weights[cisframes], -N)[-N:]
# closed state
ind_trans = np.argpartition(weights[transframes], -N)[-N:]

print(zeta[cisframes][ind_cis])
print(zeta[transframes][ind_trans])

print(SASA[cisframes][ind_cis])
print(SASA[transframes][ind_trans])



[-0.18145  -0.236303 -0.181828 -0.2074   -0.235776]
[3.082157 2.990148 2.970249 3.076367 2.955946]
[0.    0.053 0.033 0.027 0.   ]
[0.    0.06  0.027 0.    0.   ]


In [31]:
print(time[cisframes][ind_cis])
print(time[transframes][ind_trans])

[354670.016846 370830.017613 368985.017526 371875.017663 369105.017532]
[290210.013784 293995.013964 290355.013791 291210.013832 293925.013961]


In [32]:


# cis frames
for TIME in time[cisframes][ind_cis]:
    FRAME =  int(TIME/5) # 5 ps per frame
    u.trajectory[FRAME]
    protein = u.select_atoms('all')
    with md.Writer('PARK7_chainA_Pro66_cis_{}ps.pdb'.format(int(TIME)),protein.n_atoms) as W:
        W.write(protein)

        
print('Done')

Done


In [33]:


# trans frames
for TIME in time[transframes][ind_trans]:
    FRAME =  int(TIME/5) # 5 ps per frame
    u.trajectory[FRAME]
    protein = u.select_atoms('all')
    with md.Writer('PARK7_chainA_Pro66_trans_{}ps.pdb'.format(int(TIME)),protein.n_atoms) as W:
        W.write(protein)

        
print('Done')

Done
